<p style="text-align: right; margin: 0; line-height: 0;"><img width="100px" height="auto" src="../../.admin/img/eth_logo.png"></p>

<h1 style="text-align: center;">04 Mobile Algorithms</h1>
<hr>


## **2. PID Controller**

<div class="alert alert-light">

  **Name:** (add your name here)
  
  **Matriculation Number:** (add your number here)
  
</div>



### 2.1 Introduction

We already explored the fundamentals of control systems - how sensors, actuators, and controllers interact in open and closed loops to make systems behave in a desired way. One of the simplest and most widely used mechanisms for achieving this is the **PID controller** (Proportional-Integral-Derivative controller). Its role is to continuously adjust the behavior of a system so that it reaches and maintains a desired target state (the setpoint).

Continue with [05.11 Local LLM meets PID](../05_Mobile_AI/05.11_Local_LLM_meets_PID.ipynb) to use a pretrained LLM on the Raspberry Pi to explain the simulated PID response and critically compare its interpretation with measured results.


### 2.2 Feedback Loop

A control system continuously measures the output of a process, compares it to a desired reference, and adjusts its input to reduce any difference.
This closed-loop interaction between sensor, controller, and actuator allows systems to maintain stability and desired performance even when conditions change.

<center>
<img src="../../.admin/notebooks/picture/closed_loop.png" 
     alt="Closed Loop Control System" 
     width="60%" 
     style="filter: grayscale(60%);">
<p align="center"><em>Feedback (or, closed-loop) system</em></p>
</center>


We can now define PID control as a form of closed-loop control that calculates its system input from proportional, integral, and derivative terms based on the measured output. 

#### 2.2.1 Proportional Control

The simplest form of control is **proportional (P)** control. In the insulin-pump example from Notebook 04.1, glucose above the target leads the controller to increase insulin delivery; glucose below the target leads it to reduce delivery. The size of the adjustment is proportional to the current glucose error. Reducing insulin delivery does not itself supply glucose; the separate rescue input in 04.1 serves that purpose.

The interactive plots below use a generic second-order system to illustrate P, PI, and PID behavior. Its output is a position, and a positive input increases that output. Insulin has the opposite effect on glucose, so an insulin controller must use the corresponding feedback direction. Use the slider below to change the proportional gain $K_p$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from io import BytesIO


In [ ]:

def simulate_p_response(Kp=1.0):
    t = np.arange(0, 150.0, 0.005)
    dt = t[1] - t[0]
    
    setpoint_final = 10.0    # desired final position
    setpoint_time = 10.0     # time when setpoint jumps

    x = 0.0                  # position
    v = 0.0                  # velocity
    values, setpoints = [], []

    for ti in t:
        # Step input: setpoint stays 0 until setpoint_time
        setpoint = setpoint_final if ti >= setpoint_time else 0.0
        error = setpoint - x
        u = Kp * error

        # Mass-spring-damper: d²x/dt² = -x - 0.2*dx/dt + u
        dx = v
        dv = -x - 0.2*v + u
        x += dx * dt
        v += dv * dt

        values.append(x)
        setpoints.append(setpoint)

    # Plot the response
    with plt.ioff():
        fig = plt.figure(figsize=(10,4))
    plt.plot(t, values, label="System Output (x)")
    plt.plot(t, setpoints, 'r--', label="Setpoint")
    plt.xlabel("Time [s]")
    plt.ylabel("Position")
    plt.ylim(top=15.0)  # Keep the y-axis maximum fixed for every slider setting.
    plt.legend()
    plt.grid(True)
    try:
        with BytesIO() as buffer:
            fig.savefig(buffer, format="png", dpi=100)
            p_image.value = buffer.getvalue()
    finally:
        plt.close(fig)

# Retire this graph's old observers and widgets when the cell is rerun.
if "p_controls" in globals():
    for control in p_controls.values():
        control.unobserve(p_changed, names="value")
        control.close()
    p_ui.close()
    p_image.close()

p_controls = {
    "Kp": widgets.FloatSlider(value=1.0, min=0.0, max=25.0, step=0.1, description="Kp", continuous_update=False),
}
p_image = widgets.Image(format="png")
p_image.layout.max_width = "100%"

def p_changed(change=None):
    simulate_p_response(**{name: control.value for name, control in p_controls.items()})

for control in p_controls.values():
    control.observe(p_changed, names="value")
p_changed()
p_ui = widgets.VBox([*p_controls.values(), p_image])
display(p_ui)


With a small $K_p$, the correction is weak and the simulated output settles below the target. Increasing $K_p$ reduces this steady-state error, but can increase overshoot and oscillations. A P controller can only keep correcting a constant disturbance while some error remains. This is why the output in Notebook 04.1 stays slightly below the target.

To address this remaining error, we add the **integral term**.

#### 2.2.2 Proportional + Integral Control

The integral term accumulates error over time. In the insulin-pump example, if glucose remains above the target, this accumulated deviation calls for a sustained increase in insulin delivery. Once glucose reaches the target, the error stops accumulating, but the integral term retains the correction needed to maintain that balance. With suitable tuning, integral action can remove a constant steady-state error. Use the slider below to adjust the integral gain $K_i$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from io import BytesIO

def simulate_pi_response(Kp=1.0, Ki=0.0):
    t = np.arange(0, 150.0, 0.005)
    dt = t[1] - t[0]
    
    setpoint_final = 10.0   # desired final value
    setpoint_time = 10.0     # time when setpoint jumps

    x = 0.0                  # position
    v = 0.0                  # velocity
    integral = 0.0
    values, errors, integrals, setpoints = [], [], [], []

    for ti in t:
        # Step input: setpoint stays 0 until setpoint_time
        setpoint = setpoint_final if ti >= setpoint_time else 0.0
        error = setpoint - x
        integral += error * dt
        u = Kp * error + Ki * integral

        # Mass-spring-damper: d²x/dt² = -x - 0.2*dx/dt + u
        dx = v
        dv = -x - 0.2*v + u
        x += dx * dt
        v += dv * dt

        values.append(x)
        errors.append(error)
        integrals.append(integral)
        setpoints.append(setpoint)

    # Plot the response
    with plt.ioff():
        fig = plt.figure(figsize=(10,4))
    plt.plot(t, values, label="System Output (x)")
    plt.plot(t, setpoints, 'r--', label="Setpoint")
    plt.xlabel("Time [s]")
    plt.ylabel("Position")
    plt.ylim(top=15.0)  # Keep the y-axis maximum fixed for every slider setting.
    plt.legend()
    plt.grid(True)
    try:
        with BytesIO() as buffer:
            fig.savefig(buffer, format="png", dpi=100)
            pi_image.value = buffer.getvalue()
    finally:
        plt.close(fig)

# Retire this graph's old observers and widgets when the cell is rerun.
if "pi_controls" in globals():
    for control in pi_controls.values():
        control.unobserve(pi_changed, names="value")
        control.close()
    pi_ui.close()
    pi_image.close()

pi_controls = {
    "Kp": widgets.FloatSlider(value=1.0, min=0.0, max=10.0, step=0.1, description="Kp", continuous_update=False),
    "Ki": widgets.FloatSlider(value=0.0, min=0.0, max=0.2, step=0.001, description="Ki", continuous_update=False),
}
pi_image = widgets.Image(format="png")
pi_image.layout.max_width = "100%"

def pi_changed(change=None):
    simulate_pi_response(**{name: control.value for name, control in pi_controls.items()})

for control in pi_controls.values():
    control.observe(pi_changed, names="value")
pi_changed()
pi_ui = widgets.VBox([*pi_controls.values(), pi_image])
display(pi_ui)


The integral term removes the steady-state error. But if it is too strong, it can also make oscillations worse, since the controller may keep “pushing” even after the error starts changing sign. Yet, the integral term is unable to remove the oscillations. To handle that, we need one more component: the **derivative term**,  leading us to the full PID controller.

#### 2.2.3 Proportional + Integral + Derivative Control

The derivative term reacts to how quickly the error changes. In the insulin-pump example, a rapidly falling glucose measurement can make this term reduce the insulin correction before the target is crossed. It therefore responds to the current trend and can help limit overshoot and damp oscillations. Use the slider below to adjust the derivative gain $K_d$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from io import BytesIO

def simulate_pid_response(Kp=1.0, Ki=0.0, Kd=0.0):
    t = np.arange(0, 150.0, 0.005)
    dt = t[1] - t[0]
    
    setpoint_final = 10.0  # desired final value
    setpoint_time = 10.0    # time when setpoint jumps
    x = 0.0                 # position
    v = 0.0                 # velocity
    integral = 0.0
    previous_error = 0.0
    values, errors, integrals, setpoints = [], [], [], []

    for ti in t:
        # Step input: setpoint stays 0 until setpoint_time
        setpoint = setpoint_final if ti >= setpoint_time else 0.0
        error = setpoint - x
        integral += error * dt
        derivative = (error - previous_error) / dt
        u = Kp * error + Ki * integral + Kd * derivative

        # Mass-spring-damper: d²x/dt² = -x - 0.2*dx/dt + u
        dx = v
        dv = -x - 0.2*v + u
        x += dx * dt
        v += dv * dt

        values.append(x)
        errors.append(error)
        integrals.append(integral)
        setpoints.append(setpoint)
        previous_error = error

    # Plot the response
    with plt.ioff():
        fig = plt.figure(figsize=(10,4))
    plt.plot(t, values, label="System Output (x)")
    plt.plot(t, setpoints, 'r--', label="Setpoint")
    plt.xlabel("Time [s]")
    plt.ylabel("Position")
    plt.ylim(top=15.0)  # Keep the y-axis maximum fixed for every slider setting.
    plt.legend()
    plt.grid(True)
    try:
        with BytesIO() as buffer:
            fig.savefig(buffer, format="png", dpi=100)
            pid_image.value = buffer.getvalue()
    finally:
        plt.close(fig)

# Retire this graph's old observers and widgets when the cell is rerun.
if "pid_controls" in globals():
    for control in pid_controls.values():
        control.unobserve(pid_changed, names="value")
        control.close()
    pid_ui.close()
    pid_image.close()

pid_controls = {
    "Kp": widgets.FloatSlider(value=1.0, min=0.0, max=10.0, step=0.1, description="Kp", continuous_update=False),
    "Ki": widgets.FloatSlider(value=0.08, min=0.0, max=0.2, step=0.001, description="Ki", continuous_update=False),
    "Kd": widgets.FloatSlider(value=0.0, min=0.0, max=5.0, step=0.1, description="Kd", continuous_update=False),
}
pid_image = widgets.Image(format="png")
pid_image.layout.max_width = "100%"

def pid_changed(change=None):
    simulate_pid_response(**{name: control.value for name, control in pid_controls.items()})

for control in pid_controls.values():
    control.observe(pid_changed, names="value")
pid_changed()
pid_ui = widgets.VBox([*pid_controls.values(), pid_image])
display(pid_ui)


### 2.3 PID Controller

Together, the three terms form the **PID controller**:  

- **P** reacts to the current error.  
- **I** removes long-term bias by accumulating past error.  
- **D** anticipates future error and reduces oscillations.  

By tuning $(K_p, K_i, K_d)$, we can shape how the system behaves - whether it should react quickly, settle smoothly, or resist disturbances. This balance between **speed, accuracy, and stability** is why PID controllers are used everywhere, in embedded devices, drones, mobile systems, and network control.  


#### 2.3.1 PID Block Diagram  

<figure style="border: 1px solid #888; padding: 16px; margin: 16px 0;">
<img src="../../.admin/notebooks/picture/pid_feedback_diagram.svg" alt="Original PID block diagram: proportional, integral, and derivative branches sum into the system input, with negative feedback from the measured output" style="display: block; width: 100%; height: auto; margin: 0 auto;">
<figcaption style="margin-top: 12px; text-align: justify;"><strong>The target minus the measured output gives the error. The P, I, and D branches act on this same error, and their contributions are added to form the control input. The measured output returns through the negative feedback loop.</strong></figcaption>
</figure>

#### 2.3.2 PID Equation  

##### Analog PID (continuous time)

For the positive-input system used in the interactive plots, the controller output is calculated as:  

$$
u(t) = K_p \, e(t) + K_i \int_0^t e(\tau)\, d\tau + K_d \frac{de(t)}{dt}
$$

where:  
- $e(t)$ = setpoint − measured value (the error)  
- $K_p, K_i, K_d$ = proportional, integral, derivative gains  

##### Digital PID (discrete time)

For a digital controller with a fixed sampling interval $\Delta t$, let $e_n=r_n-y_n$ be the error at sample $n$, where $r_n$ is the setpoint and $y_n$ is the measured value. Approximating the integral by a sum and the derivative by a backward difference gives

$$
u_n = K_p e_n + K_i\,\Delta t\sum_{j=0}^{n} e_j + K_d\,\frac{e_n-e_{n-1}}{\Delta t}.
$$

The same equation can be implemented without storing every past error:

$$
I_n = I_{n-1} + e_n\,\Delta t, \qquad
D_n = \frac{e_n-e_{n-1}}{\Delta t}, \qquad
u_n = K_p e_n + K_i I_n + K_d D_n.
$$

Here, $I_n$ is the accumulated error and $D_n$ is its estimated rate of change. The interactive simulations above use these updates with `dt` as the sampling interval. They initialize the accumulated error and previous error to zero ($I_{-1}=0$ and $e_{-1}=0$).



### 2.4 Choosing the PID Gains

Choosing $K_p$, $K_i$, and $K_d$ is called **PID tuning**. There is no single best combination for every system. The choice depends on the desired balance between response speed, overshoot, remaining error, and control effort. Improving one of these can worsen another.

#### 2.4.1 Basic Manual Tuning

Manual tuning usually begins with proportional control alone, with $K_i=K_d=0$. A gradual increase in $K_p$ strengthens the response and reduces the remaining error, but can also increase overshoot and oscillations. A useful starting point is a reasonably fast response without excessive oscillation.

Integral action is then introduced gradually. It accumulates the remaining error and can bring the output to the target. Too much integral action can increase overshoot and settling time, so $K_i$ and $K_p$ may need adjustment together.

Derivative action can help damp the response. A gradual increase in $K_d$ may reduce overshoot and oscillation, followed by further adjustment of all three gains. Derivative action is sensitive to sensor noise, so practical controllers often filter it.

The result is judged by how far the output overshoots, how long it takes to settle near the target, and whether an offset remains. Settling time means the time after the target change until the output enters and stays within a chosen tolerance around the target. A response clipped by the fixed y-axis range should not be mistaken for a well-behaved response.

This procedure provides a starting point, rather than a proof of stability. The plots here illustrate a target change; a complete evaluation would also consider disturbances, noise, and actuator limits. For an insulin-pump application, the direction and delay of insulin's effect, the sampling interval, and delivery limits also affect the gain choice.

#### 2.4.2 Automated and AI-Assisted Tuning

Automated tuning evaluates many gain combinations against a score that can penalize tracking error, overshoot, long settling time, and excessive control effort. Unstable responses are rejected. The chosen gains depend on how these goals are weighted and should be evaluated across several operating conditions.

**AI can assist this search**, for example through a learned model that predicts which gain combinations are worth evaluating. AI is optional: numerical optimization and established model-based tuning methods can also choose gains. [MathWorks' PID Tuner documentation](https://www.mathworks.com/help/control/ref/pidtuner-app.html) describes automatic tuning that balances performance and robustness; it is an example of automated tuning without requiring AI.

When an external search chooses fixed gains, the controller still applies the same PID equations. A system that changes its gains during operation is an adaptive controller and needs additional validation. Understanding manual tuning helps in judging whether automatically proposed gains produce acceptable behavior.


### 2.5 An Alternative: Neural-Network Control

AI can do more than choose PID gains: a **neural-network controller** can learn the control rule itself. A PID controller multiplies its three error signals by fixed gains and adds the results. A neural controller instead passes these signals through a network with trainable weights to produce the control input:

$$
u_n=f_\theta\!\left(e_n,\ I_n,\ \frac{e_n-e_{n-1}}{\Delta t}\right).
$$

Here, $f_\theta$ is the network and $\theta$ represents its learned weights. The neural-control example on the course-polish-2026 branch uses a small network with eight hidden units and a bounded output. It receives the same types of error information as PID, so the difference is how that information is converted into a control action.

**How training works.** The network controls a simulated system across different targets and starting conditions. A loss measures tracking error, motion, final error, and control effort over the whole response. Backpropagation through the network and simulator determines how the weights affect this loss; an optimizer such as Adam updates them. Repeating this process aims to improve the response. The neural-control paper, PyTorch automatic-differentiation documentation, and Adam paper in the references provide further background.

After training, the weights can remain fixed during operation. This differs from continuously learning while controlling the system, and from using AI only to tune the three PID gains. A trained network can express a nonlinear control rule, but it is not automatically better than PID. Its behavior depends on the simulator and training conditions. Fair comparisons use the same available measurements and actuator limits, and test unfamiliar targets, loads, delays, and disturbances. A lower training loss alone does not establish stability or reliable behavior under these changes.


### 2.6 Regulatory Acceptance

PID controllers are considered deterministic systems, which is one of the main reasons they are trusted and accepted in medical devices and other safety-critical technologies. A PID controller produces an output that depends directly and transparently on measurable quantities: the current error, the accumulated error over time, and the rate of change of the error. In other words, for any given sequence of inputs, the controller’s output will always be the same. There is no randomness, no adaptive learning process, and no hidden internal dynamics that might cause unexpected behavior. This determinism makes the controller fully predictable and mathematically analyzable.

Another important reason for regulatory acceptance is the transparency of the PID design. The controller has only three parameters-proportional, integral, and derivative gains-and each has a clear physical meaning. Engineers can explain exactly why each parameter has a certain value, how it was tuned, and what effect it has on the system. There is no “black box” behavior, which makes validation, documentation, and certification straightforward.

When a PID controller malfunctions or is poorly tuned, its failure modes are also well understood: the system may oscillate, overshoot, or drift, but it will not exhibit chaotic or unpredictable responses. This predictability allows additional safety mechanisms and fault detection systems to be built around it. 

For these reasons, regulatory agencies generally favor deterministic and analyzable control systems. PID controllers remain a standard solution in many critical applications, such as insulin pumps and artificial pancreas systems, mechanical ventilators, infusion pumps, and industrial process controls. Even though more advanced control or learning methods exist, PID controllers are still preferred wherever safety, reliability, and traceability are more important than optimization. They are accepted because they are deterministic, interpretable, and their behavior can be rigorously proven - a combination that makes regulatory approval possible.

### 2.7 Reading Material and References

The following readings are available in the course's [Background Reading collection](https://gitlab.ethz.ch/wireless/BackgroundReading/-/blob/main/Algorithms%20and%20Game%20Theory/PID%20Controller/README.md). Read the PID tutorial first; the book chapters and robot paper provide further detail.

- [Tim Wescott (2016), *PID Without a PhD* (revised tutorial)](https://gitlab.ethz.ch/wireless/BackgroundReading/-/blob/main/Algorithms%20and%20Game%20Theory/PID%20Controller/2016-pid-without-a-phd.pdf) - Start here for a practical explanation of the P, I, and D terms, implementation in software, and controller tuning.
- [Tim Wescott (2006), *Applied Control Theory for Embedded Systems*, Chapter 1: The Basics](https://gitlab.ethz.ch/wireless/BackgroundReading/-/blob/main/Algorithms%20and%20Game%20Theory/PID%20Controller/Applied-Control-Theory-for-Embedded-Systems/2006-01-the-basics.pdf) - Background on feedback and control systems in embedded devices.
- [Tim Wescott (2006), *Applied Control Theory for Embedded Systems*, Chapter 7: Sampling Theory](https://gitlab.ethz.ch/wireless/BackgroundReading/-/blob/main/Algorithms%20and%20Game%20Theory/PID%20Controller/Applied-Control-Theory-for-Embedded-Systems/2006-07-sampling-theory.pdf) - How a digital controller samples measurements and interacts with a continuous physical process; complements the discrete-time equation in Section 2.3.2.
- [Tim Wescott (2006), *Applied Control Theory for Embedded Systems*, Chapter 10: Software Implications](https://gitlab.ethz.ch/wireless/BackgroundReading/-/blob/main/Algorithms%20and%20Game%20Theory/PID%20Controller/Applied-Control-Theory-for-Embedded-Systems/2006-10-software-implications.pdf) - Practical implementation issues, including numerical precision and computation time.
- [*Balancing Control of Bicycle Robot Using PID Control* (2013)](https://gitlab.ethz.ch/wireless/BackgroundReading/-/blob/main/Algorithms%20and%20Game%20Theory/PID%20Controller/2013-balancing-bicycle-robot-pid.pdf) - An application of PID feedback to balancing a bicycle robot using a reaction wheel.

**Video introductions**

- [PID Controllers Explained: Why P-Only Control Isn't Enough](https://gitlab.ethz.ch/wireless/BackgroundReading/-/blob/main/Algorithms%20and%20Game%20Theory/PID%20Controller/README.md#further-reading-and-resources) - A video introduction to PID controllers and why proportional control alone may be insufficient.
- [What Is PID Control? | Understanding PID Control, Part 1](https://gitlab.ethz.ch/wireless/BackgroundReading/-/blob/main/Algorithms%20and%20Game%20Theory/PID%20Controller/README.md#further-reading-and-resources) - An introductory video on PID control, complementing the explanations of feedback and the three controller terms.

The video links above open the remote Background Reading PID guide, where you can follow the YouTube links.

**Control Theory and Neural Control References**

- [Åström, K. J., & Murray, R. M. (2021), *Feedback Systems: An Introduction for Scientists and Engineers, Second Edition*](https://fbswiki.org/wiki/index.php/Main_Page) - Open textbook.
- [Narendra, K. S., & Parthasarathy, K. (1990), *Identification and Control of Dynamical Systems Using Neural Networks*](https://doi.org/10.1109/72.80202) - IEEE Transactions on Neural Networks, 1(1), 4-27.
- [PyTorch, *Automatic Differentiation with torch.autograd*](https://docs.pytorch.org/tutorials/beginner/basics/autogradqs_tutorial.html) - Documentation.
- [Kingma, D. P., & Ba, J. (2015), *Adam: A Method for Stochastic Optimization*](https://arxiv.org/abs/1412.6980) - ICLR.


----
### 2.8 Hands-On Exercises

<div class="alert alert-block alert-info">Fill in your responses and save this notebook. See the <a href="../../03_Assignments/Submission-Guidelines.md">submission guidelines</a> for how to submit your work.</div>



**Tune the controller using the three sliders in the full PID example (Section 2.2.3). Start with $K_i=K_d=0$ and adjust $K_p$. Then add $K_i$ to reduce the remaining error, and finally adjust $K_d$ to reduce oscillations. Record one gain combination at each stage (P, PI, and PID), and describe how the remaining error, overshoot, and settling time change. Aim for a response close to the target of 10 with little overshoot and no persistent oscillation.**

Answer:



**Which final gain combination did you choose, and why is its balance of response speed, overshoot, and remaining error acceptable?**

Answer:



**What happens to the simulated output if you set $K_p$ very small?**

Answer:



**What do you observe when $K_p$ is very large?**

Answer:



**How does the integral term fix the main weakness of proportional control?**

Answer:



**Why can a very large $K_i$ make the system overshoot and oscillate?**

Answer:



**How does the derivative term improve the response compared to PI control?**

Answer:



**In your own words, explain what each term (P, I, D) cares about: present, past, or future error.**

Answer:



**OPTIONAL: Did you read through and understand all the tasks in this Notebook? Are there any shortcomings, missing parts, or sections that should be improved?**

Answer: